In [1]:
import pickle
import torch
import torch.optim as optim
import copy
import random
import os

random.seed(42)
device = 'cuda' if torch.cuda.is_available() else 'cpu'

In [2]:
!git clone https://github.com/GiovanniAdelfio/small_LM
%cd small_LM

Cloning into 'small_LM'...
remote: Enumerating objects: 499, done.
remote: Counting objects: 100% (137/137), done.
remote: Compressing objects: 100% (123/123), done.
remote: Total 499 (delta 70), reused 11 (delta 11), pack-reused 362 (from 2)
Receiving objects: 100% (499/499), 10.80 MiB | 13.29 MiB/s, done.
Resolving deltas: 100% (230/230), done.
/content/small_LM


In [3]:
path = os.getcwd() + os.sep
path_files = path + "files" + os.sep

In [4]:
!gdown 1GVZg5tTCFUwXhRl6gW6ZQKk5NHle2zZe

Downloading...
From (original): https://drive.google.com/uc?id=1GVZg5tTCFUwXhRl6gW6ZQKk5NHle2zZe
From (redirected): https://drive.google.com/uc?id=1GVZg5tTCFUwXhRl6gW6ZQKk5NHle2zZe&confirm=t&uuid=728a45da-1897-4578-a9c6-d94614747530
To: /content/small_LM/R-gpt-epoch=35-val_loss=2.7651.ckpt
100% 27.3M/27.3M [00:01<00:00, 25.8MB/s]


## Downloads

In [5]:
import torch

In [6]:
from tokenizers import Tokenizer

tokenizer = Tokenizer.from_file(path_files + "custom_tokenizer_tinystories.json")

In [7]:
train_dataset, val_dataset, test_dataset = torch.load(path_files + "dataset_riddles.pt", weights_only= "True")

In [8]:
from torch.utils.data import DataLoader
from utils.data import SLM_dataset
bs = 32
cs = 128

train = SLM_dataset(train_dataset, cs)
val = SLM_dataset(val_dataset, cs)

train_dataloader = DataLoader(train, batch_size=bs, shuffle=True, num_workers=0)
val_dataloader = DataLoader(val, batch_size=bs, shuffle=False, num_workers=0)

/content/small_LM/utils/data.py:56: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  masks.append(torch.tensor(padding_mask, dtype=torch.bool))


#Inference

In [9]:
!pip install lightning

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 5.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 853.6/853.6 kB 57.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 983.4/983.4 kB 78.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 857.3/857.3 kB 68.4 MB/s eta 0:00:00


We import our model, and generation function. We then initialize the model.

In [10]:
import torch
from model.lightning_model import GPTLightningModule

model = GPTLightningModule(
    block_size=128,
    vocab_size=4000,
    n_embd=256,
    n_head=8,
    n_layer=6,
    use_qat=False,  # Default
    learning_rate = 1e-3,
    dropout = 0.1,
    weight_decay=1e-1
)

checkpoint = torch.load(path + "R-gpt-epoch=35-val_loss=2.7651.ckpt")
model.load_state_dict(checkpoint)

model.eval();
model.to("cuda" if torch.cuda.is_available() else "cpu")

GPTLightningModule(
  (model): GPTModel(
    (token_embedding_table): Embedding(4000, 256)
    (position_embedding_table): Embedding(128, 256)
    (emb_dropout): Dropout(p=0.1, inplace=False)
    (blocks): Sequential(
      (0): Block(
        (mha): MultiheadAttention(
          (out_proj): NonDynamicallyQuantizableLinear(in_features=256, out_features=256, bias=True)
        )
        (ffwd): FeedFoward(
          (net): Sequential(
            (0): Linear(in_features=256, out_features=1024, bias=True)
            (1): ReLU()
            (2): Linear(in_features=1024, out_features=256, bias=True)
            (3): Dropout(p=0.1, inplace=False)
          )
        )
        (ln1): LayerNorm((256,), eps=1e-05, elementwise_affine=True)
        (ln2): LayerNorm((256,), eps=1e-05, elementwise_affine=True)
        (dropout1): Dropout(p=0.1, inplace=False)
        (dropout2): Dropout(p=0.1, inplace=False)
      )
      (1): Block(
        (mha): MultiheadAttention(
          (out_proj): NonDyn

In [11]:
#Input
question = "Tell me a riddle about the "
topic = "sea"
encoded = tokenizer.encode("[BOS] " + question + topic + " \n")
input_ids = torch.tensor(encoded.ids)
input_ids = input_ids.reshape(1, -1)

In [15]:
out = model.generate(input_ids.cuda(), 300, temperature=0.4, top_k=50, repetition_penalty=1.2)
testo_generato = tokenizer.decode(out[0].tolist(), skip_special_tokens= False)

separatore = "[EOS]"
if separatore in testo_generato:
    testo_pulito = testo_generato[5:].split(separatore)[0]
else:
    testo_pulito = testo_generato

print(testo_pulito)

 Tell me a riddle about the sea 
 RIDDLE: I'm the only one of my strength and I'll never be seen. And I can't see the sea, but I have no eyes or no legs. 
 ANSWER: beach 


In [17]:
from utils.inference import ValutatoreIndovinelli
valutatore = ValutatoreIndovinelli()

topic_richiesto = topic
testo_output = testo_pulito

print("\n--- REPORT DI VALUTAZIONE ---")

# TEST STRUTTURA
struttura_ok, riddle, answer = valutatore.check_struttura(testo_output)
print(f"Struttura Rispettata: {'✅ SI' if struttura_ok else '❌ NO'}")

if struttura_ok:
    # TEST RIPETITIVITÀ (N-Grams)
    # Sopra 0.3 sospetto. Sopra 0.5: loop
    ripetitività = valutatore.calcola_ripetitivita(riddle)
    print(f"Tasso di Ripetitività: {ripetitività} {'✅ (Ottimo)' if ripetitività < 0.2 else '⚠️ (Attenzione)'}")

    # TEST COERENZA
    coerenza = valutatore.check_coerenza_contesto(topic_richiesto, answer, riddle)
    if coerenza == 2:
        print("Coerenza Contesto: 🟢 Perfetta (Risposta centrata)")
    elif coerenza == 1:
        print("Coerenza Contesto: 🟡 Media (Topic menzionato)")
    else:
        print("Coerenza Contesto: 🔴 Fuori Tema")

    # TEST INGLESE
    # Meno di 20/30: Inglese eccellente. Tra 30 e 80: Normale/Accettabile. Oltre 100: Senza senso.
    perplessita = valutatore.calcola_perplessita(testo_output)
    print(f"Perplexity (Fluidità testuale): {perplessita}")

Inizializzazione del valutatore...


Loading weights:   0%|          | 0/56 [00:00<?, ?it/s]

GPTNeoForCausalLM LOAD REPORT from: roneneldan/TinyStories-33M
Key                                                   | Status     |  | 
------------------------------------------------------+------------+--+-
transformer.h.{0, 1, 2, 3}.attn.attention.bias        | UNEXPECTED |  | 
transformer.h.{0, 1, 2, 3}.attn.attention.masked_bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.



--- REPORT DI VALUTAZIONE ---
Struttura Rispettata: ✅ SI
Tasso di Ripetitività: 0.0 ✅ (Ottimo)
Coerenza Contesto: 🟡 Media (Topic menzionato)
Perplexity (Fluidità testuale): 72.61
